# Grupa A – Logistic Regression

Sprawdźcie, jak liczba i jakość informacji wpływają na przewidywanie założenia lokaty. Notebook sam przygotowuje dane i pipeline; nie trzeba ręcznie czyścić ani kodować kolumn.

`y = yes` oznacza założenie lokaty. `duration` jest wyłączone, bo przed rozmową nie jest znane.

**Colab:** uruchomcie komórkę klonowania poniżej. **Lokalnie w VS Code:** pomińcie ją.

In [ ]:
# Uruchom tylko w Google Colab; lokalnie w VS Code pomiń tę komórkę.
!git clone https://github.com/poochacz/Podstawy_technologii_ML.git /content/eda_ml_workshop_repo

In [1]:
import sys
from pathlib import Path

COLAB_ROOT = Path("/content/eda_ml_workshop_repo/02_ml_bank_marketing_colab")
current = Path.cwd().resolve()
candidates = [COLAB_ROOT]
for directory in (current, *current.parents):
    candidates.extend((directory / "02_ml_bank_marketing_colab", directory))

ROOT = next(
    (candidate.resolve() for candidate in candidates if (candidate / "scripts").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Nie znaleziono folderu 02_ml_bank_marketing_colab.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
print("Środowisko gotowe:", ROOT)

Środowisko gotowe: C:\Users\dariu\Documents\Studia Kato\Koźmiński\podstawy ML\podstawy_ML_repo\Podstawy_technologii_ML\02_ml_bank_marketing_colab


In [2]:
from scripts.data_setup import prepare_model_data

N_RECORDS = 2000
RANDOM_STATE = 42
df, data_metadata = prepare_model_data(
    n=N_RECORDS,
    random_state=RANDOM_STATE,
    data_dir=DATA_DIR,
)
print(f"Dane gotowe: {df.shape[0]} rekordów, {df.shape[1] - 1} cech.")

Dane gotowe: 2000 rekordów, 9 cech.


## Eksperyment: liczba i jakość danych

Zmieniajcie `TRAIN_SIZES` lub `REPEATS`. Każdy rozmiar używa zagnieżdżonej próbki treningowej, a test pozostaje taki sam. `degraded` oznacza 15% losowo ukrytych wartości w treningu; test nie jest zmieniany.

In [3]:
import pandas as pd
from IPython.display import display
from scripts.modeling import (
    always_no_baseline,
    run_experiments,
    save_experiment_results,
    summarize_experiments,
)

TRAIN_SIZES = (100, 300, 800, 1500)
REPEATS = 3
MODEL_NAME = "logistic_regression"

baseline = always_no_baseline(df, split_seed=RANDOM_STATE)
runs = run_experiments(
    data=df,
    model_name=MODEL_NAME,
    train_sizes=TRAIN_SIZES,
    repeats=REPEATS,
    split_seed=RANDOM_STATE,
)
summary = summarize_experiments(runs)
results_path = save_experiment_results(summary, MODEL_NAME, DATA_DIR)
display(pd.DataFrame([baseline]))
display(summary)
print(f"Podsumowanie zapisano w: {results_path}")

,model,test_rows,accuracy,recall_yes
0,zawsze brak lokaty,400,0.8875,0.0


,model,feature_set,train_size,accuracy_mean,accuracy_std,recall_yes_mean,recall_yes_std
0,logistic_regression,basic,100,0.887,0.000,0.000,0.000
1,logistic_regression,basic,300,0.887,0.001,0.000,0.000
2,logistic_regression,basic,800,0.887,0.000,0.000,0.000
3,logistic_regression,basic,1500,0.887,0.000,0.000,0.000
4,logistic_regression,degraded,100,0.872,0.013,0.074,0.013
5,logistic_regression,degraded,300,0.883,0.006,0.081,0.013
6,logistic_regression,degraded,800,0.884,0.003,0.059,0.026
7,logistic_regression,degraded,1500,0.878,0.001,0.067,0.000
8,logistic_regression,standard,100,0.881,0.005,0.074,0.013
9,logistic_regression,standard,300,0.884,0.006,0.067,0.022


Podsumowanie zapisano w: C:\Users\dariu\Documents\Studia Kato\Koźmiński\podstawy ML\podstawy_ML_repo\Podstawy_technologii_ML\02_ml_bank_marketing_colab\data\results\logistic_regression_summary.csv


In [ ]:
from scripts.modeling import plot_metric

plot_metric(summary, metric="recall_yes")

## Jak działa Logistic Regression?

Dodatni współczynnik wiąże się z większym wynikiem `yes`, ujemny z mniejszym. To związek użyty przez model, nie dowód przyczynowości.

In [ ]:
from scripts.modeling import fit_explanation_model
from scripts.explainability import plot_logistic_coefficients

explanation_model, X_test, y_test = fit_explanation_model(
    df,
    model_name=MODEL_NAME,
    feature_set="standard",
    train_size=1500,
    split_seed=RANDOM_STATE,
)
coefficients = plot_logistic_coefficients(explanation_model, top_n=12)
display(coefficients)

## Pytania do dyskusji

1. Czy większa próbka zawsze poprawiała wynik w podobnym stopniu?
2. Czy wynik dla 100 rekordów był stabilny między powtórzeniami?
3. Jak różnił się `basic` od `standard` i `degraded`?
4. Czy współczynnik dowodzi, że dana cecha powoduje założenie lokaty?